# SEATauBench – chạy thử agent mô hình nhỏ trên tiếng Việt (pilot)

**Mục tiêu notebook:** đo nhanh xem một mô hình nhỏ chạy cục bộ (mặc định `Qwen/Qwen3-4B-Instruct-2507`) làm được bao nhiêu task chăm sóc khách hàng tiếng Việt trong SEATauBench, mất bao lâu và tốn bao nhiêu tiền API cho *người dùng giả lập*. Kết quả dùng để quyết định có chọn đề tài này hay không.

**Notebook làm 4 việc:**
1. Cài SEATauBench (bản mở rộng τ²-bench cho 5 ngôn ngữ Đông Nam Á, có tiếng Việt).
2. Đo **mức sàn**: một agent “giả” luôn từ chối mọi yêu cầu được bao nhiêu điểm. *Không cần GPU, không cần API.*
3. Dựng mô hình nhỏ bằng vLLM trên GPU Kaggle và chạy thử 5 task (tiếng Anh vs tiếng Việt).
4. Tổng hợp kết quả + ước tính thời gian/chi phí cho lần chạy đầy đủ.

**Thiết lập Kaggle trước khi chạy:**
- *Settings → Accelerator:* **GPU T4 x2**
- *Settings → Internet:* **On**
- *Add-ons → Secrets:* thêm `OPENROUTER_API_KEY` (dùng cho người dùng giả lập Qwen3-235B, giống bài báo gốc). Bước 2 không cần key.

> Ghi chú: lĩnh vực **airline** chấm bằng trạng thái CSDL + thông tin phải truyền đạt, **không cần LLM giám khảo**, nên rẻ nhất để chạy thử. Retail cần thêm LLM giám khảo (NL assertions).

## 0. Cấu hình

In [ ]:
import os, json, subprocess, time, shutil, signal
from pathlib import Path

WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
REPO_DIR = WORK / 'seataubench'

# --- Mô hình agent (chạy cục bộ bằng vLLM) ---
AGENT_MODEL   = 'Qwen/Qwen3-4B-Instruct-2507'   # đổi sang model khác để so sánh (vd. Qwen/Qwen3-1.7B, Qwen/Qwen3-8B)
VLLM_PORT     = 8000
MAX_MODEL_LEN = 24576                             # chính sách + mô tả công cụ + hội thoại dài; giảm nếu thiếu VRAM

# --- Người dùng giả lập (API) ---
USER_LLM      = 'openrouter/qwen/qwen3-235b-a22b-2507'   # giống bài báo gốc để kết quả so được

# --- Phạm vi chạy thử ---
DOMAIN        = 'airline'
SPLIT         = 'test'        # airline: train 30 / test 20 task
NUM_TASKS     = 5
NUM_TRIALS    = 1
SCENARIOS     = ['english', 'l2_domain']   # thêm 'l2_interaction', 'l2_tools' khi chạy thật
LANG          = 'vi'
CONCURRENCY   = 3

# Thành phần được Việt hoá theo từng kịch bản (theo README của SEATauBench)
SCENARIO_COMPONENTS = {
    'english':        [],
    'l2_tools':       ['tools'],
    'l2_interaction': ['user_system', 'agent_system', 'greeting'],
    'l2_domain':      ['user_system', 'agent_system', 'greeting', 'tools', 'policy', 'db', 'tasks'],
}
print('WORK =', WORK)

## 1. Cài SEATauBench
Dự án yêu cầu Python 3.12+, nên dùng `uv` để tự tải Python 3.12 vào môi trường riêng (không đụng tới Python của Kaggle).

In [ ]:
!pip install -q uv
if not REPO_DIR.exists():
    !git clone --depth 1 https://github.com/SEACrowd/SEATauBench {REPO_DIR}
%cd {REPO_DIR}
!uv sync --python 3.12 --extra experiments 2>&1 | tail -3

# Mô hình nhận diện ngôn ngữ fastText (dùng cho chỉ số 'agent có trả lời đúng ngôn ngữ không')
!mkdir -p data/models
if not Path('data/models/lid.176.bin').exists():
    !curl -sL -o data/models/lid.176.bin https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin
!uv run tau2 run --help | head -3

## 2. Hàm chạy và hàm tổng hợp kết quả

In [ ]:
from math import comb
import pandas as pd

def run_tau2(scenario, agent_llm, agent_args, user_llm, user_args, save_to,
             domain=DOMAIN, split=SPLIT, num_tasks=NUM_TASKS, trials=NUM_TRIALS,
             concurrency=CONCURRENCY, extra_env=None):
    """Gọi CLI `tau2 run` của SEATauBench cho một kịch bản."""
    lang = 'en' if scenario == 'english' else LANG
    cmd = ['uv', 'run', 'tau2', 'run', '--domain', domain,
           '--seatau-scenario', scenario, '--lang-id', lang,
           '--agent-llm', agent_llm, '--agent-llm-args', json.dumps(agent_args),
           '--user-llm', user_llm, '--user-llm-args', json.dumps(user_args),
           '--task-split-name', split, '--num-trials', str(trials),
           '--max-concurrency', str(concurrency), '--save-to', save_to]
    if num_tasks:
        cmd += ['--num-tasks', str(num_tasks)]
    comps = SCENARIO_COMPONENTS[scenario]
    if comps:
        cmd += ['--lang-components', *comps]
    env = {**os.environ, **(extra_env or {})}
    t0 = time.time()
    p = subprocess.run(cmd, cwd=REPO_DIR, env=env, capture_output=True, text=True)
    if p.returncode != 0:
        print(p.stdout[-3000:]); print(p.stderr[-3000:])
        raise RuntimeError(f'tau2 run thất bại ({scenario})')
    print(f'✓ {scenario}: xong sau {time.time()-t0:.0f}s → data/simulations/{save_to}')
    return REPO_DIR / 'data' / 'simulations' / save_to / 'results.json'

def pass_hat_k(df, k):
    """pass^k của τ-bench: xác suất làm đúng cả k lần chạy của cùng một task."""
    vals = []
    for _, g in df.groupby('task_id'):
        n, c = len(g), int((g.reward >= 1).sum())
        if n >= k:
            vals.append(comb(c, k) / comb(n, k))
    return sum(vals) / len(vals) if vals else float('nan')

def load_runs(path):
    r = json.load(open(path))
    rows = []
    for s in r['simulations']:
        msgs = s.get('messages') or []
        rows.append({
            'task_id': str(s['task_id']), 'trial': s.get('trial'),
            'reward': (s.get('reward_info') or {}).get('reward', 0.0),
            'turns': sum(1 for m in msgs if m.get('role') in ('assistant', 'user')),
            'tool_calls': sum(len(m.get('tool_calls') or []) for m in msgs if m.get('role') == 'assistant'),
            'duration_s': s.get('duration'),
            'agent_cost': s.get('agent_cost') or 0.0, 'user_cost': s.get('user_cost') or 0.0,
            'termination': s.get('termination_reason'),
        })
    return pd.DataFrame(rows)

def summarize(paths):
    out = []
    for name, p in paths.items():
        df = load_runs(p)
        out.append({'run': name, 'n_sims': len(df), 'pass^1': round(pass_hat_k(df, 1), 3),
                    'avg_turns': round(df.turns.mean(), 1), 'avg_tool_calls': round(df.tool_calls.mean(), 1),
                    'avg_sec/sim': round(df.duration_s.mean(), 1),
                    'user_cost_$/sim': round(df.user_cost.mean(), 4),
                    'termination': dict(df.termination.value_counts())})
    return pd.DataFrame(out)

## 3. Mức sàn: agent “luôn từ chối” (không cần GPU, không cần API)

Một máy chủ giả trả lời theo kiểu OpenAI: agent chỉ chào, tra cứu một lần rồi từ chối; người dùng giả nói một câu rồi kết thúc. Agent này **không bao giờ thay đổi CSDL**, nên nó “đúng” ở mọi task mà đáp án là *không làm gì*.

Mọi kết quả của mô hình thật nên được so với mức sàn này, không chỉ với 0.

In [ ]:
MOCK = WORK / 'mock_refuse_server.py'
MOCK.write_text('''
import json, time
from http.server import BaseHTTPRequestHandler, HTTPServer
class H(BaseHTTPRequestHandler):
    def log_message(self, *a): pass
    def do_GET(self):
        self.send_response(200); self.send_header("Content-Type", "application/json"); self.end_headers()
        self.wfile.write(json.dumps({"data": [{"id": "mock"}]}).encode())
    def do_POST(self):
        body = json.loads(self.rfile.read(int(self.headers["Content-Length"])))
        model, n = body.get("model", ""), len(body.get("messages", []))
        if "user" in model:
            msg = {"role": "assistant", "content": "Tôi cần hỗ trợ đặt chỗ." if n < 4 else "###STOP###"}
        elif body.get("tools") and n < 4:
            msg = {"role": "assistant", "content": None, "tool_calls": [{"id": "call_1", "type": "function",
                   "function": {"name": "get_user_details", "arguments": json.dumps({"user_id": "unknown"})}}]}
        else:
            msg = {"role": "assistant", "content": "Xin lỗi, tôi không thể thực hiện yêu cầu này."}
        out = json.dumps({"id": "x", "object": "chat.completion", "created": int(time.time()), "model": model,
            "choices": [{"index": 0, "message": msg, "finish_reason": "tool_calls" if msg.get("tool_calls") else "stop"}],
            "usage": {"prompt_tokens": 1, "completion_tokens": 1, "total_tokens": 2}}).encode()
        self.send_response(200); self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(out))); self.end_headers(); self.wfile.write(out)
HTTPServer(("127.0.0.1", 8011), H).serve_forever()
''')
mock_proc = subprocess.Popen(['python', str(MOCK)]); time.sleep(2)
mock_args = {'temperature': 0, 'api_base': 'http://127.0.0.1:8011/v1'}

floor_paths = {}
for sc in SCENARIOS:
    for split in ['train', 'test']:
        floor_paths[f'floor_{sc}_{split}'] = run_tau2(
            sc, 'hosted_vllm/mock-agent', mock_args, 'hosted_vllm/mock-user', mock_args,
            save_to=f'floor_{DOMAIN}_{sc}_{split}', split=split, num_tasks=None, concurrency=8)
mock_proc.terminate()
summarize(floor_paths)[['run', 'n_sims', 'pass^1']]

*Kết quả kiểm tra trước (22/10/2026, airline):* agent luôn từ chối đạt **pass^1 ≈ 0,46** trên cả 50 task (0,53 ở train, 0,35 ở test), như nhau cho tiếng Anh và tiếng Việt. Để so sánh, bài báo gốc báo cáo GPT-5-mini ≈ 0,54 và Qwen3-235B ≈ 0,49 ở airline tiếng Việt. Lưu ý: người dùng giả ở đây dừng rất sớm; với người dùng giả thật con số có thể khác đôi chút.

## 4. Dựng mô hình nhỏ bằng vLLM
vLLM được cài vào một môi trường `uv` riêng để không làm hỏng PyTorch của Kaggle. GPU T4 không hỗ trợ bf16 nên dùng `--dtype half`.

In [ ]:
VLLM_ENV = WORK / 'vllm-env'
if not (VLLM_ENV / 'bin' / 'vllm').exists():
    !uv venv {VLLM_ENV} --python 3.12 -q
    !uv pip install --python {VLLM_ENV}/bin/python -q vllm
n_gpu = len(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip().splitlines())
print('Số GPU:', n_gpu)

vllm_log = open(WORK / 'vllm.log', 'w')
vllm_proc = subprocess.Popen([
    str(VLLM_ENV / 'bin' / 'vllm'), 'serve', AGENT_MODEL,
    '--port', str(VLLM_PORT), '--dtype', 'half',
    '--max-model-len', str(MAX_MODEL_LEN), '--gpu-memory-utilization', '0.90',
    '--tensor-parallel-size', str(max(1, n_gpu)),
    '--enable-auto-tool-choice', '--tool-call-parser', 'hermes',   # parser gọi công cụ cho dòng Qwen
], stdout=vllm_log, stderr=subprocess.STDOUT)

import urllib.request
for i in range(120):   # chờ tối đa ~20 phút (lần đầu phải tải model)
    try:
        urllib.request.urlopen(f'http://127.0.0.1:{VLLM_PORT}/v1/models', timeout=5); print('vLLM sẵn sàng'); break
    except Exception:
        if vllm_proc.poll() is not None:
            print(open(WORK / 'vllm.log').read()[-3000:]); raise RuntimeError('vLLM dừng bất thường')
        time.sleep(10)
else:
    raise TimeoutError('vLLM chưa lên sau 20 phút – xem vllm.log')

Kiểm tra nhanh: mô hình có gọi công cụ đúng định dạng không (nếu bước này hỏng thì mọi task sẽ hỏng).

In [ ]:
import urllib.request
req = {'model': AGENT_MODEL, 'temperature': 0,
       'messages': [{'role': 'user', 'content': 'Tra cứu thông tin người dùng có mã emma_kim_9957 giúp tôi.'}],
       'tools': [{'type': 'function', 'function': {'name': 'get_user_details', 'description': 'Lấy thông tin người dùng',
                  'parameters': {'type': 'object', 'properties': {'user_id': {'type': 'string'}}, 'required': ['user_id']}}}]}
r = urllib.request.urlopen(urllib.request.Request(f'http://127.0.0.1:{VLLM_PORT}/v1/chat/completions',
        data=json.dumps(req).encode(), headers={'Content-Type': 'application/json'}), timeout=120)
print(json.dumps(json.loads(r.read())['choices'][0]['message'], ensure_ascii=False, indent=1))

## 5. Chạy thử: tiếng Anh vs tiếng Việt
Người dùng giả lập gọi qua OpenRouter (cần secret `OPENROUTER_API_KEY`).

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['OPENROUTER_API_KEY'] = UserSecretsClient().get_secret('OPENROUTER_API_KEY')
except Exception as e:
    print('Không đọc được secret:', e)
assert os.environ.get('OPENROUTER_API_KEY'), 'Thiếu OPENROUTER_API_KEY'

agent_llm  = f'hosted_vllm/{AGENT_MODEL}'
agent_args = {'temperature': 0, 'api_base': f'http://127.0.0.1:{VLLM_PORT}/v1'}
user_args  = {'temperature': 0}
tag = AGENT_MODEL.split('/')[-1]

pilot_paths = {}
for sc in SCENARIOS:
    pilot_paths[f'{tag}_{sc}'] = run_tau2(sc, agent_llm, agent_args, USER_LLM, user_args,
                                          save_to=f'pilot_{DOMAIN}_{tag}_{sc}_{SPLIT}{NUM_TASKS}')
summary = summarize(pilot_paths)
summary

## 6. So với mức sàn + ước tính cho lần chạy đầy đủ

In [ ]:
floor_test = {sc: pass_hat_k(load_runs(floor_paths[f'floor_{sc}_test']), 1) for sc in SCENARIOS}
for _, row in summary.iterrows():
    sc = row['run'].split(f'{tag}_')[-1]
    print(f"{row['run']:40s} pass^1 = {row['pass^1']:.3f}   (mức sàn 'luôn từ chối' trên test: {floor_test[sc]:.3f})")
print('Lưu ý: chỉ 5 task nên sai số rất lớn; đây là số để ước lượng, không phải kết quả.')

# Ước tính cho chạy thật: 50 task airline × 3 lần × 3 kịch bản
full_sims = 50 * 3 * 3
sec_per_sim = summary['avg_sec/sim'].mean() / CONCURRENCY
cost_per_sim = summary['user_cost_$/sim'].mean()
print(f'Ước tính {full_sims} hội thoại: ~{full_sims*sec_per_sim/3600:.1f} giờ GPU, ~${full_sims*cost_per_sim:.2f} tiền người dùng giả lập')

## 7. Xem một hội thoại (để hiểu agent sai ở đâu)

In [ ]:
def show(path, idx=0, max_chars=300):
    s = json.load(open(path))['simulations'][idx]
    print('Task', s['task_id'], '| reward', (s.get('reward_info') or {}).get('reward'), '|', s['termination_reason'])
    for m in s['messages']:
        if m.get('tool_calls'):
            for tc in m['tool_calls']:
                print(f"  [{m['role']} → gọi {tc['name']}] {json.dumps(tc.get('arguments'), ensure_ascii=False)[:max_chars]}")
        if m.get('content'):
            print(f"  [{m['role']}] {m['content'][:max_chars]}")

show(pilot_paths[f'{tag}_l2_domain'], idx=0)

## 8. Lưu kết quả và tắt server

In [ ]:
shutil.make_archive(str(WORK / 'seatau_pilot_results'), 'zip', REPO_DIR / 'data' / 'simulations')
summary.to_csv(WORK / 'pilot_summary.csv', index=False)
vllm_proc.send_signal(signal.SIGINT)
print('Đã lưu: seatau_pilot_results.zip, pilot_summary.csv trong /kaggle/working')